## memo

- URL: https://www.kaggle.com/code/koyamaryuji/qwen-array-task-inference/notebook?scriptVersionId=342918010
- multi candidate trace v2
- ood

In [11]:
import polars as pl
from pathlib import Path
import sys
sys.path.append(str('d:/qwen_reasoning_test/ArrayTask'))
from src.gen_task import RAW_RULES, RAW_OOD_RULES_V1
RULES = [
    {
        "id": f"{i:03d}",
        "primitives": rule,
    }
    for i, rule in enumerate(RAW_OOD_RULES_V1)
]

df = pl.read_csv(Path("debug_predictions.csv"))
df = df.sort("id")
stop = 0
total_token = 0
for pred in df.iter_rows(named=True):
    print("###" * 50)
    print(f'Task ID: \n{pred["id"].encode().decode("unicode-escape")}')
    print("===" * 50)
    print(f'Prompt: \n{pred["raw_prompt"].encode().decode("unicode-escape")}')
    print("===" * 50)
    print(f'Raw_output: \n{pred["raw_output"]}')
    print("===" * 50)
    print(f'Target: \n{pred["target"]}')
    print("===" * 50)
    print(f'Finish: \n{pred["finish_reason"]}')
    print("===" * 50)
    print(f'tokens: \n{pred["num_tokens"]}')
    total_token += pred["num_tokens"]
    if pred["finish_reason"] == "stop":
        stop += 1

print(stop)
print(f"mean_token: {total_token / len(df)}")

######################################################################################################################################################
Task ID: 
000_0
Prompt: 

Infer the transformation rule from the examples.

State the rule you inferred in one short sentence.
Then apply the rule to the Query input.
Finally, output the complete final array.
        
Example 1

Input:
[1, 0, 4, 3, 3, 2, 1, 8, 1, 9]

Output:
[0, 9, 1, 0, 4, 3, 3, 2, 1, 8, 1]

Example 2

Input:
[0, 0, 1, 3, 3, 8, 9, 0]

Output:
[0, 0, 0, 0, 1, 3, 3, 8, 9]

Example 3

Input:
[3, 8, 6, 3, 7, 9, 4, 0, 2]

Output:
[0, 2, 3, 8, 6, 3, 7, 9, 4, 0]

Example 4

Input:
[6, 5, 4, 2, 3, 5, 1, 1, 6, 1]

Output:
[0, 1, 6, 5, 4, 2, 3, 5, 1, 1, 6]

Example 5

Input:
[5, 9, 4, 0, 7, 8, 1]

Output:
[0, 1, 5, 9, 4, 0, 7, 8]

Query

Input:
[1, 8, 4, 9, 5, 9, 3, 1]

Output:
Raw_output: 
<think>
Candidate rule: ['shift_right', 'prepend_zeros']

Example 0:
Input: [1, 0, 4, 3, 3, 2, 1, 8, 1, 9]
Result: [0, 0, 1, 0, 4, 3, 3, 2, 1

In [17]:
import re
import ast
import polars as pl
from pathlib import Path


def extract_answer(text):
    if text is None:
        return 'NOT_FOUND'

    matches = re.findall(r'\[[^\[\]]*\]', text)
    arrays = []
    for match in matches:
        try:
            value = ast.literal_eval(match)

            if isinstance(value, list):
                arrays.append(value)

        except (ValueError, SyntaxError):
            pass

    if arrays == []:
        return 'NOT_FOUND'

    return str(arrays[-1])

df = pl.read_csv(Path("debug_predictions.csv"))

match_count = 0
match = []
miss = []
for pred in df.iter_rows(named=True):
    # print("###" * 50)
    answer = extract_answer(pred["raw_output"])
    if answer == pred["target"]:
        # print(pred["id"])
        match_count += 1
        match.append(pred["id"].split("_")[0])
    else:
        miss.append(pred["id"].split("_")[0])
        # print(pred["id"])
        # print(pred["raw_prompt"].encode().decode("unicode-escape"))
        # print(pred["target"])
print(f"len(df): {len(df)}")
print(f"match_count: {match_count}")
print(f"acc: {match_count / len(df)}")
# print(miss)
from collections import Counter


counts = Counter(miss)
match_counts = Counter(match)

len(df): 540
match_count: 128
acc: 0.23703703703703705


# 正解

In [7]:
match_results = []

for rule in RULES:
    if rule["id"] in [i for i, j in match_counts.items()]:
        for i, j in match_counts.items():
            if rule["id"] == i:
                match_results.append({"task_id": i, "count": j, "rule": rule["primitives"]})
    else:
        match_results.append({"task_id": rule["id"], "count": 0, "rule": rule["primitives"]})

match_results = sorted(match_results, key=lambda x: x["task_id"], reverse=True)

for x in match_results:
    print(x)

{'task_id': '035', 'count': 6, 'rule': ['swap', 'take_odd_positions']}
{'task_id': '034', 'count': 2, 'rule': ['swap', 'take_even_positions']}
{'task_id': '033', 'count': 1, 'rule': ['swap', 'mirror']}
{'task_id': '032', 'count': 1, 'rule': ['swap', 'pairwise_sum']}
{'task_id': '031', 'count': 0, 'rule': ['swap', 'differences']}
{'task_id': '030', 'count': 2, 'rule': ['swap', 'modulo']}
{'task_id': '029', 'count': 1, 'rule': ['swap', 'add_constant']}
{'task_id': '028', 'count': 3, 'rule': ['swap', 'multiply_constant']}
{'task_id': '027', 'count': 6, 'rule': ['swap', 'pop_left']}
{'task_id': '026', 'count': 1, 'rule': ['swap', 'pop_right']}
{'task_id': '025', 'count': 1, 'rule': ['swap', 'append_zeros']}
{'task_id': '024', 'count': 3, 'rule': ['swap', 'prepend_zeros']}
{'task_id': '023', 'count': 2, 'rule': ['rotate_left', 'take_odd_positions']}
{'task_id': '022', 'count': 4, 'rule': ['rotate_left', 'take_even_positions']}
{'task_id': '021', 'count': 8, 'rule': ['rotate_left', 'mirror']

# 不正解

In [8]:
results = []
for i, j in counts.items():
    for rule in RULES:
        if rule["id"] == i:
            # print(i, j, rule)
            results.append({"task_id": i, "count": j, "rule": rule["primitives"]})

results = sorted(results, key=lambda x: x["task_id"], reverse=True)
results

[{'task_id': '035', 'count': 9, 'rule': ['swap', 'take_odd_positions']},
 {'task_id': '034', 'count': 13, 'rule': ['swap', 'take_even_positions']},
 {'task_id': '033', 'count': 14, 'rule': ['swap', 'mirror']},
 {'task_id': '032', 'count': 14, 'rule': ['swap', 'pairwise_sum']},
 {'task_id': '031', 'count': 15, 'rule': ['swap', 'differences']},
 {'task_id': '030', 'count': 13, 'rule': ['swap', 'modulo']},
 {'task_id': '029', 'count': 14, 'rule': ['swap', 'add_constant']},
 {'task_id': '028', 'count': 12, 'rule': ['swap', 'multiply_constant']},
 {'task_id': '027', 'count': 9, 'rule': ['swap', 'pop_left']},
 {'task_id': '026', 'count': 14, 'rule': ['swap', 'pop_right']},
 {'task_id': '025', 'count': 14, 'rule': ['swap', 'append_zeros']},
 {'task_id': '024', 'count': 12, 'rule': ['swap', 'prepend_zeros']},
 {'task_id': '023',
  'count': 13,
  'rule': ['rotate_left', 'take_odd_positions']},
 {'task_id': '022',
  'count': 11,
  'rule': ['rotate_left', 'take_even_positions']},
 {'task_id': '02